# HW2 reproduction: Shavrina's HW2 code, run as written, with two changes

This notebook runs **her HW2 scripts** (`embedding_encoder.py` for stage A, `reranker_encoder.py` for stage B) on the starter protocol. The modules are written out exactly as in her transcript. Only two things change:

1. **Leakage fix (approved):** the customer profile uses `PROFILE_RULE` (default `approved`, which requires `lastQuestionnaireDate < cutoff`, not the placeholder date, and row `timestamp < cutoff`). Her code used the timestamp-only rule.
2. **Same-information control (added):** `reranker_profile_onehot_knn` gives the reranker the same profile fields as the LLM text, without the LLM. Its neighbour score and rank are the only extra features.

Nothing else is changed. In particular the universe, row policy, training windows (March, April, May, with the stale 2021-03-01 embedding), candidate sources and UMAP layout are hers.

**Runtime per method** comes from her manifest (`*_train_predict_seconds` for stage B, `*_reduce_seconds` and `*_knn_seconds` for stage A). Shared feature building is reported separately.

**Verification:** the notebook recomputes nDCG@10 independently from the raw CSVs for every starter user with a future purchase, and prints the comparison.

**Run two settings:** `MAX_USERS = 100` (the HW2 table) and `MAX_USERS = 100000` (the full warm cohort, for cohort analysis). The full run is much slower.

In [3]:
!pip -q install -U "transformers>=5.2" flash-linear-attention lightgbm umap-learn scikit-learn

In [4]:
import os, sys, json, hashlib, subprocess, time, shutil, platform, importlib.metadata as md_
def _ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

CUTOFF = '2021-06-01'
END = '2021-07-01'
HORIZON_DAYS = 30
PROFILE_RULE = 'approved'            # 'approved' (leakage fix) or 'timestamp' (Shavrina's HW2 rule)
MAX_USERS = 100                      # 100 = HW2 table; 100000 = full warm cohort
MODEL = 'Qwen/Qwen3.5-0.8B'
MODEL_REVISION = '2fc06364715b967f1860aea9cf38778875588b17'
MAX_LEN, BATCH = 384, 64
REPO_COMMIT = 'f685086'
TAG_DATES = ['2021-03-01', '2021-06-01']   # her embedding files: training (stale) and test
DIMS = (128, 32, 8, 2)
assert PROFILE_RULE in ('approved', 'timestamp')

DATA_DIR = Path(os.environ.get('FAR_TRANS_DIR', '/kaggle/input/datasets/tresjudamen/far-trans'))
WORK = Path(os.environ.get('WORK_DIR', '/kaggle/working'))
REPO = WORK / 'RecSys-LLMs'
SRC = WORK / 'src'
HW2 = WORK / 'hw2' / 'far-trans-llm'
STARTER = WORK / 'hw2' / 'far-trans'
EMB = WORK / 'outputs' / 'emb'
RUN_TAG = f'{PROFILE_RULE}_{MAX_USERS}'
OUT_A = WORK / 'outputs' / f'stageA_encoder_{RUN_TAG}'
OUT_B = WORK / 'outputs' / f'stageB_reranker_{RUN_TAG}'
for p in (SRC, HW2, EMB, WORK / 'hw2' / 'outputs'):
    p.mkdir(parents=True, exist_ok=True)
assert (DATA_DIR / 'transactions.csv').exists(), f'Set FAR_TRANS_DIR (now: {DATA_DIR})'
print('data files:', sorted(p.name for p in DATA_DIR.iterdir()))
print('run:', RUN_TAG, '| outputs:', OUT_A.name, OUT_B.name)

data files: ['README.txt', 'asset_information.csv', 'close_prices.csv', 'customer_information.csv', 'limit_prices.csv', 'markets.csv', 'questionnaires.csv', 'transactions.csv']
run: approved_100 | outputs: stageA_encoder_approved_100 stageB_reranker_approved_100


In [5]:
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/dryjins/RecSys-LLMs.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', REPO_COMMIT], check=True)
if not STARTER.exists():
    shutil.copytree(REPO / 'project' / 'far-trans', STARTER)
for f in (REPO / 'project' / 'far-trans-llm').glob('*.py'):
    shutil.copy(f, HW2 / f.name)
print('starter:', sorted(p.name for p in STARTER.iterdir()))
print('far-trans-llm:', sorted(p.name for p in HW2.iterdir()))

Cloning into '/kaggle/working/RecSys-LLMs'...


starter: ['.gitignore', 'ANNOUNCEMENT.md', 'README.md', 'extensions.py', 'instructions', 'starter.py', 'test_starter.py']
far-trans-llm: ['data_adapter.py', 'encoder.py', 'ranker.py', 'runtime.py', 'test_encoder.py', 'test_ranker.py']


In [6]:
MODULES = {
'src/data.py': r'''
"""FAR-Trans loading, temporal split and evaluation.

Everything a model may see is computed from rows dated strictly before the
cut-off; the evaluation window [cutoff, end) is used only for ground truth.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

import numpy as np
import pandas as pd


def load(data_dir: str) -> dict[str, pd.DataFrame]:
    r = lambda n: pd.read_csv(os.path.join(data_dir, n))
    tx = r('transactions.csv')
    tx['ts'] = pd.to_datetime(tx.timestamp)
    ai = r('asset_information.csv')
    ai['ts'] = pd.to_datetime(ai.timestamp)
    ci = r('customer_information.csv')
    ci['ts'] = pd.to_datetime(ci.timestamp)
    cp = r('close_prices.csv')
    cp['ts'] = pd.to_datetime(cp.timestamp)
    return {'tx': tx, 'assets': ai, 'customers': ci, 'prices': cp}


@dataclass
class Split:
    """One temporal split. `items` is the eligible universe at the cut-off."""
    cutoff: pd.Timestamp
    end: pd.Timestamp
    items: list[str]
    item_ix: dict[str, int]
    past: pd.DataFrame            # all transactions (Buy and Sell) before cut-off, eligible items only
    truth: dict[str, set[int]]    # user -> eligible items bought in the window
    warm: list[str]               # evaluated users with >= 1 pre-cut-off buy
    cold: list[str]               # evaluated users without one (handled by teammate; popularity fallback here)
    assets: pd.DataFrame = field(repr=False)   # attributes known at cut-off, indexed by ISIN, in item order
    prices: pd.DataFrame = field(repr=False)   # close prices before cut-off


def make_split(d: dict, cutoff: str, end: str) -> Split:
    t0, te = pd.Timestamp(cutoff), pd.Timestamp(end)
    pr = d['prices'][d['prices'].ts < t0]
    a = d['assets'][d['assets'].ts < t0].sort_values('ts').drop_duplicates('ISIN', keep='last')
    items = sorted(set(pr.ISIN) & set(a.ISIN))           # priced and described before cut-off
    ix = {s: i for i, s in enumerate(items)}
    tx = d['tx'][d['tx'].ISIN.isin(ix)]
    past = tx[tx.ts < t0]
    fut = tx[(tx.ts >= t0) & (tx.ts < te) & (tx.transactionType == 'Buy')]
    truth = {u: {ix[s] for s in g.ISIN} for u, g in fut.groupby('customerID')}
    buyers = set(past[past.transactionType == 'Buy'].customerID)
    warm = sorted(u for u in truth if u in buyers)
    cold = sorted(u for u in truth if u not in buyers)
    return Split(t0, te, items, ix, past, truth, warm, cold,
                 a.set_index('ISIN').loc[items], pr)


_DISC = 1.0 / np.log2(np.arange(2, 12))


def ndcg10(ranked: list[int], rel: set[int]) -> float:
    dcg = sum(_DISC[i] for i, it in enumerate(ranked[:10]) if it in rel)
    return dcg / _DISC[:min(len(rel), 10)].sum()


def check_top10(ranked, n_items: int) -> None:
    assert len(ranked) == 10 and len(set(ranked)) == 10, 'need ten unique items'
    assert all(0 <= i < n_items for i in ranked), 'item outside eligible universe'


def evaluate(recs: dict[str, list[int]], sp: Split, users: list[str]) -> np.ndarray:
    out = []
    for u in users:
        check_top10(recs[u], len(sp.items))
        out.append(ndcg10(recs[u], sp.truth[u]))
    return np.array(out)


def bootstrap_ci(x: np.ndarray, n: int = 10_000, alpha: float = 0.05, seed: int = 0):
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, len(x), (n, len(x)))].mean(1)
    return float(np.quantile(m, alpha / 2)), float(np.quantile(m, 1 - alpha / 2))
''',
'src/features.py': r'''
"""Signals, candidate generation and (user, item) features for one split.

All quantities use only `Split.past` (before the cut-off).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from data import Split

ATTRS = ['assetCategory', 'assetSubCategory', 'sector', 'industry', 'marketID']


def _l2(m):
    return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)


RISK = {'Conservative': 0, 'Income': 1, 'Balanced': 2, 'Aggressive': 3}
CAP = {'CAP_LT30K': 0, 'CAP_30K_80K': 1, 'CAP_80K_300K': 2, 'CAP_GT300K': 3}


def ials(B: np.ndarray, conf: np.ndarray, k: int = 64, reg: float = 100.0, iters: int = 12, seed: int = 0):
    # reg/alpha/k tuned under the paper protocol on non-test dates (scripts/tune_ials.py):
    # reg=0.1 overfits (nDCG@10 ~0.04 on new assets); reg=100, alpha=10, decayed confidence is best
    """Implicit ALS (Hu, Koren, Volinsky 2008) in numpy. B: binary preferences (U x N),
    conf: confidence weights c_ui - 1 on the non-zeros. Returns user and item factors."""
    rng = np.random.default_rng(seed)
    U, N = B.shape
    P, Q = rng.normal(0, 0.01, (U, k)), rng.normal(0, 0.01, (N, k))
    nz_u = [np.flatnonzero(B[u]) for u in range(U)]
    nz_i = [np.flatnonzero(B[:, i]) for i in range(N)]
    I = reg * np.eye(k)

    def solve(F, nz, c_rows):
        G = F.T @ F + I
        out = np.zeros((len(nz), k))
        for r, idx in enumerate(nz):
            if len(idx) == 0:
                continue
            c = c_rows(r, idx)
            Fi = F[idx]
            out[r] = np.linalg.solve(G + (Fi.T * c) @ Fi, Fi.T @ (1 + c))
        return out

    for _ in range(iters):
        P = solve(Q, nz_u, lambda r, idx: conf[r, idx])
        Q = solve(P, nz_i, lambda r, idx: conf[idx, r])
    return P.astype(np.float32), Q.astype(np.float32)


def _ord(v, m):
    return m.get(str(v).replace('Predicted_', ''), -1)


class Signals:
    def __init__(self, sp: Split, ease_lambda: float = 200.0, customers: pd.DataFrame | None = None,
                 mf: bool = True):
        self.sp = sp
        t0, N = sp.cutoff, len(sp.items)
        buys = sp.past[sp.past.transactionType == 'Buy']
        sells = sp.past[sp.past.transactionType == 'Sell']
        self.users = sorted(buys.customerID.unique())
        self.uix = {u: i for i, u in enumerate(self.users)}
        U = len(self.users)
        ui = buys.customerID.map(self.uix).values
        ii = buys.ISIN.map(sp.item_ix).values
        age = (t0 - buys.ts).dt.days.values.astype(np.float32)

        def acc(w):
            m = np.zeros((U, N), np.float32); np.add.at(m, (ui, ii), w); return m

        self.X = acc(1.0)                          # buy counts
        self.Xd90 = acc(0.5 ** (age / 90))         # time-decayed counts
        self.Xd365 = acc(0.5 ** (age / 365))
        self.Xd30 = acc(0.5 ** (age / 30))
        # net units still held = bought - sold (FAR-Trans has no opening positions; approximate)
        self.units = acc(buys.units.values.astype(np.float32))
        self.val = acc(buys.totalValue.values.astype(np.float32))
        self.last = np.full((U, N), 9999.0, np.float32)
        np.minimum.at(self.last, (ui, ii), age)
        self.first = np.zeros((U, N), np.float32)
        np.maximum.at(self.first, (ui, ii), age)
        su = sells.customerID.map(self.uix); ok = su.notna().values
        self.S = np.zeros((U, N), np.float32)
        np.add.at(self.S, (su.values[ok].astype(int), sells.ISIN.map(sp.item_ix).values[ok]), 1.0)
        np.subtract.at(self.units, (su.values[ok].astype(int), sells.ISIN.map(sp.item_ix).values[ok]),
                       sells.units.values[ok].astype(np.float32))
        self.held = (self.units > 1e-6).astype(np.float32)
        self.B = (self.X > 0).astype(np.float32)
        self.u_last = np.where(self.X > 0, self.last, 9999).min(1)
        self.u_n = self.X.sum(1)

        # item popularity (distinct buyers) over several windows
        def pop(days):
            b = buys[buys.ts >= t0 - pd.Timedelta(days=days)]
            p = np.zeros(N, np.float32)
            np.add.at(p, b.ISIN.map(sp.item_ix).values, 1.0)
            return p
        self.pop = {d: pop(d) for d in (30, 90, 180, 100000)}
        self.trend = (self.pop[30] + 1) / (self.pop[180] / 6 + 1)

        # price features from close prices before cut-off
        px = sp.prices.pivot_table(index='ts', columns='ISIN', values='closePrice').reindex(columns=sp.items).ffill()
        lastp = px.iloc[-1].values
        def ret(days):
            ref = px[px.index <= t0 - pd.Timedelta(days=days)]
            base = ref.iloc[-1].values if len(ref) else np.full(N, np.nan)
            return np.nan_to_num(lastp / base - 1, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        self.ret = {d: ret(d) for d in (30, 90, 365)}
        daily = px[px.index > t0 - pd.Timedelta(days=90)].pct_change()
        self.vol = np.nan_to_num(daily.std().values, nan=0.0).astype(np.float32)

        # content: one-hot attributes (classical CBF of the course slides)
        A = sp.assets[ATTRS].fillna('unknown').astype(str)
        self.F = _l2(pd.get_dummies(A).values.astype(np.float32))
        self.cat_codes = {c: pd.factorize(A[c].values)[0].astype(int) for c in ATTRS[:3]}

        # collaborative: item-kNN cosine and EASE
        n = np.sqrt(self.B.sum(0)) + 1e-9
        self.S_item = (self.B.T @ self.B) / n[:, None] / n[None, :]
        np.fill_diagonal(self.S_item, 0)
        G = self.B.T @ self.B + ease_lambda * np.eye(N, dtype=np.float32)
        P = np.linalg.inv(G)
        self.W_ease = (-P / np.diag(P)[None, :]).astype(np.float32)
        np.fill_diagonal(self.W_ease, 0)

        # collaborative: implicit ALS on decayed purchase confidence
        self.mf = None
        if mf:
            Pf, Qf = ials(self.B, 10 * np.log1p(self.Xd365 * 4))
            self.mf = Pf @ Qf.T

        # investor profile at the cut-off and risk fit (suitability-style features)
        U = len(self.users)
        self.u_risk = np.full(U, -1, np.int8)
        self.u_cap = np.full(U, -1, np.int8)
        self.u_type = np.full(U, -1, np.int8)
        self.u_risk_pred = np.zeros(U, np.int8)
        if customers is not None:
            c = customers[customers.ts < t0].sort_values('ts').drop_duplicates('customerID', keep='last')
            c = c.set_index('customerID').reindex(self.users)
            self.u_risk = np.array([_ord(v, RISK) for v in c.riskLevel], np.int8)
            self.u_cap = np.array([_ord(v, CAP) for v in c.investmentCapacity], np.int8)
            self.u_type = pd.Categorical(
                c.customerType.fillna('unknown'),
                categories=['Mass', 'Premium', 'Professional', 'Inactive', 'Legal Entity', 'unknown']).codes.astype(np.int8)
            self.u_risk_pred = c.riskLevel.astype(str).str.startswith('Predicted_').values.astype(np.int8)
        # mean risk level of each item's past buyers, and popularity within each risk segment (180 days)
        known = self.u_risk >= 0
        w = self.B[known]
        self.i_risk = (self.u_risk[known] @ w) / (w.sum(0) + 1e-9)
        recent = (self.last < 180).astype(np.float32)
        self.pop_seg = {}
        for lvl in range(-1, 4):
            m = self.u_risk == lvl
            p = recent[m].sum(0)
            self.pop_seg[lvl] = p / (p.sum() + 1e-9)

    # ---- representation-based user-kNN ---------------------------------
    def userknn(self, R: np.ndarray, users: list[str], k: int = 50, metric: str = 'cos') -> dict[str, np.ndarray]:
        """Score items by the purchases of the k nearest users in representation R
        (rows aligned with self.users). Neighbours' purchases are weighted by similarity."""
        out = {}
        Rn = _l2(R) if metric == 'cos' else R
        for u in users:
            q = Rn[self.uix[u]]
            sims = Rn @ q if metric == 'cos' else 1.0 / (1.0 + np.linalg.norm(Rn - q, axis=1))
            sims[self.uix[u]] = -np.inf
            nn = np.argpartition(-sims, k)[:k]
            out[u] = sims[nn] @ self.Xd365[nn]
        return out


def candidates(sg: Signals, users, knn: dict, n=50, new_only=False) -> dict[str, np.ndarray]:
    """Union of top-n per source: own past buys, user-kNN, item-kNN, EASE, iALS, 90-day popularity.
    new_only: drop assets the user bought before (paper protocol)."""
    top = lambda s, k: np.argpartition(-s, min(k, len(s) - 1))[:k]
    p90 = sg.pop[90]
    out = {}
    for u in users:
        r = sg.uix[u]
        b = sg.B[r]
        c = set() if new_only else set(np.flatnonzero(b))
        srcs = [knn[u], b @ sg.S_item, b @ sg.W_ease, p90]
        if sg.mf is not None:
            srcs.append(sg.mf[r])
        for s in srcs:
            s = np.where(b > 0, -np.inf, s) if new_only else s
            c |= set(top(s, n))
        if new_only:
            c -= set(np.flatnonzero(b))
        out[u] = np.array(sorted(c))
    return out


def feature_frame(sg: Signals, users, cands, knn_feats: dict[str, dict]) -> pd.DataFrame:
    """One row per (user, candidate). knn_feats: name -> {user: score vector}."""
    rows = []
    for u in users:
        r, c = sg.uix[u], cands[u]
        x, b = sg.X[r], sg.B[r]
        prof = x @ sg.F
        f = {
            'user': u, 'item': c,
            'n_buy': x[c], 'dec90': sg.Xd90[r, c], 'dec365': sg.Xd365[r, c],
            'days_last': sg.last[r, c], 'days_first': sg.first[r, c],
            'n_sell': sg.S[r, c], 'net_buys': x[c] - sg.S[r, c],
            'dec30': sg.Xd30[r, c], 'held': sg.held[r, c], 'u_n_held': np.full(len(c), sg.held[r].sum()),
            'val_share': sg.val[r, c] / (sg.val[r].sum() + 1e-9),
            'u_n': np.full(len(c), sg.u_n[r]), 'u_distinct': np.full(len(c), b.sum()),
            'u_days_last': np.full(len(c), sg.u_last[r]),
            'pop30': sg.pop[30][c], 'pop90': sg.pop[90][c], 'pop180': sg.pop[180][c],
            'pop_all': sg.pop[100000][c], 'trend': sg.trend[c],
            'ret30': sg.ret[30][c], 'ret90': sg.ret[90][c], 'ret365': sg.ret[365][c], 'vol90': sg.vol[c],
            'cbf': sg.F[c] @ prof / (np.linalg.norm(prof) + 1e-9),
            'itemknn': (b @ sg.S_item)[c], 'ease': (b @ sg.W_ease)[c],
            'u_risk': np.full(len(c), sg.u_risk[r]), 'u_cap': np.full(len(c), sg.u_cap[r]),
            'u_type': np.full(len(c), sg.u_type[r]), 'u_risk_pred': np.full(len(c), sg.u_risk_pred[r]),
            'i_risk': sg.i_risk[c], 'risk_gap': sg.i_risk[c] - sg.u_risk[r],
            'pop_seg': sg.pop_seg[int(sg.u_risk[r])][c],
        }
        for nm, s in (('itemknn', b @ sg.S_item), ('ease', b @ sg.W_ease)) + (
                (('mf', sg.mf[r]),) if sg.mf is not None else ()):
            if nm == 'mf':
                f['mf'] = s[c]
            f[nm + '_rank'] = (-s).argsort().argsort()[c]
        for a, codes in sg.cat_codes.items():
            share = np.bincount(codes, weights=x, minlength=codes.max() + 1) / (x.sum() + 1e-9)
            f[f'share_{a}'] = share[codes[c]]
            f[f'code_{a}'] = codes[c]
        for name, sc in knn_feats.items():
            s = sc[u]
            f[name] = s[c]
            f[name + '_rank'] = (-s).argsort().argsort()[c]
        rows.append(pd.DataFrame(f))
    return pd.concat(rows, ignore_index=True)
''',
'src/embed.py': r'''
"""Frozen-LLM investor embeddings (prompted encoder, zero-shot, no weight updates).

For each split cut-off, every user with a pre-cut-off buy is serialised as text from
information dated before the cut-off only, then encoded:

  --model Qwen/Qwen3.5-0.8B          generative LLM; prompt-based last-token embedding
                                     (PromptEOL: '... means in one word:"')
  --model Qwen/Qwen3-Embedding-0.6B  dedicated embedding model; last-token pooling

Texts are deduplicated across cut-offs (inactive users' texts repeat), so each unique
text is encoded once. Output: outputs/emb/<tag>_<cutoff>.npy + .users.txt

python src/embed.py --model Qwen/Qwen3.5-0.8B --tag qwen35 --cutoffs 2022-06-01 2021-12-01 ...
"""
from __future__ import annotations

import argparse
import gzip
import hashlib
import json
import os
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(__file__))
from data import load, make_split  # noqa: E402

MODEL_REVISION = '2fc06364715b967f1860aea9cf38778875588b17'   # Qwen/Qwen3.5-0.8B snapshot used in the HW2 runs


def profile_at(ci: pd.DataFrame, cutoff: pd.Timestamp) -> dict:
    c = ci[ci.ts < cutoff].sort_values('ts').drop_duplicates('customerID', keep='last')
    return c.set_index('customerID').to_dict('index')


def user_texts(sp, ci) -> dict[str, str]:
    t0 = sp.cutoff
    prof = profile_at(ci, t0)
    a = sp.assets
    past = sp.past.copy()
    past['age'] = (t0 - past.ts).dt.days
    b = past[past.transactionType == 'Buy'].assign(w=lambda x: 0.5 ** (x.age / 365))
    n_sell = past[past.transactionType == 'Sell'].groupby('customerID').size().to_dict()
    ub = b.groupby('customerID').agg(n=('age', 'size'), last=('age', 'min'))
    agg = (b.groupby(['customerID', 'ISIN']).agg(n=('age', 'size'), last=('age', 'min'), w=('w', 'sum'))
           .reset_index().sort_values(['customerID', 'w'], ascending=[True, False]))
    agg['cat'] = a.assetCategory.fillna('unknown').reindex(agg.ISIN).values
    n_assets = agg.groupby('customerID').size().to_dict()
    catmix = (b.assign(cat=a.assetCategory.fillna('unknown').reindex(b.ISIN).values)
              .groupby('customerID').cat.value_counts(normalize=True))
    desc = {i: ', '.join(str(v) for v in (x.assetCategory, x.assetSubCategory, x.sector, x.industry)
                         if isinstance(v, str)) for i, x in a.iterrows()}
    name = a.assetName.to_dict()
    top = {u: g.head(12)[['ISIN', 'n', 'last']].values.tolist() for u, g in agg.groupby('customerID', sort=False)}
    mix = {u: s for u, s in catmix.groupby(level=0)}
    out = {}
    for u, r in ub.iterrows():
        p = prof.get(u)
        if p:
            risk = p['riskLevel']
            src = 'estimated' if str(risk).startswith('Predicted_') else 'questionnaire'
            head = (f"Customer type {p['customerType']}; risk level {str(risk).replace('Predicted_', '')} ({src}); "
                    f"investment capacity {p['investmentCapacity']}.")
        else:
            head = 'Profile unknown.'
        lines = [head,
                 f"{int(r.n)} buys and {n_sell.get(u, 0)} sells of {n_assets[u]} assets; "
                 f"last buy {int(r['last'])} days ago.",
                 'Category mix: ' + ', '.join(f'{k[1]} {v:.0%}' for k, v in mix[u].items()) + '.',
                 'Assets bought (most weight first):']
        for i, n, last in top[u]:
            lines.append(f"- {name[i]} ({desc[i]}): {int(n)} buys, last {int(last)} days ago")
        out[u] = '\n'.join(lines)
    return out


class Encoder:
    def __init__(self, model: str, max_len: int, threads: int):
        import torch
        from transformers import AutoModel, AutoTokenizer
        torch.set_num_threads(threads)
        self.torch = torch
        self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.tok = AutoTokenizer.from_pretrained(model, revision=MODEL_REVISION)
        self.tok.padding_side = 'left'
        dtype = torch.float16 if self.dev == 'cuda' else torch.float32
        self.m = AutoModel.from_pretrained(model, revision=MODEL_REVISION, dtype=dtype).to(self.dev).eval()
        self.max_len = max_len
        self.prompt_eol = 'Embedding' not in model

    def wrap(self, t: str) -> str:
        if self.prompt_eol:     # zero-shot prompt that makes the last token summarise the investor
            return f'This investor record:\n"{t}"\nmeans in one word:"'
        return f'Instruct: Represent this investor to find investors with similar future purchases\nQuery: {t}'

    def __call__(self, texts: list[str], bs: int) -> np.ndarray:
        out = []
        with self.torch.inference_mode():
            for i in range(0, len(texts), bs):
                enc = self.tok([self.wrap(t) for t in texts[i:i + bs]], padding=True, truncation=True,
                               max_length=self.max_len, return_tensors='pt').to(self.dev)
                h = self.m(**enc, output_hidden_states=False)
                h = h.last_hidden_state if hasattr(h, 'last_hidden_state') else h[0]
                out.append(h[:, -1].float().cpu().numpy())   # left padding: last position is the last token
        return np.concatenate(out)


TEXTS = 'texts.json.gz'      # {"unique": {hash: text}, "cutoffs": {cutoff: {user: hash}}}


def export(args):
    """Local, CPU: build the leakage-free texts for every cut-off."""
    d = load(args.data)
    per_cut, uniq = {}, {}
    for c in args.cutoffs:
        sp = make_split(d, c, str((pd.Timestamp(c) + pd.DateOffset(months=1)).date()))
        tx = user_texts(sp, d['customers'])
        per_cut[c] = {}
        for u, t in tx.items():
            h = hashlib.md5(t.encode()).hexdigest()
            uniq.setdefault(h, t)
            per_cut[c][u] = h
        print(f'{c}: {len(tx)} users', flush=True)
    print(f'unique texts: {len(uniq)}', flush=True)
    with gzip.open(os.path.join(args.out, TEXTS), 'wt', encoding='utf-8') as f:
        json.dump({'unique': uniq, 'cutoffs': per_cut}, f)
    u, h = next(iter(per_cut[args.cutoffs[0]].items()))
    open(os.path.join(args.out, f'example_text_{args.cutoffs[0]}.txt'), 'w', encoding='utf-8').write(
        f'user {u}\n\n{uniq[h]}\n')


def encode(args):
    """GPU recommended (Colab/Kaggle): encode every unique text once; resumable."""
    with gzip.open(os.path.join(args.out, TEXTS), 'rt', encoding='utf-8') as f:
        uniq = json.load(f)['unique']
    cache_f = os.path.join(args.out, f'{args.tag}_cache.npz')
    cache = dict(np.load(cache_f)) if os.path.exists(cache_f) else {}
    todo = sorted((k for k in uniq if k not in cache), key=lambda k: len(uniq[k]))
    enc = Encoder(args.model, args.max_len, args.threads)
    print(f'device {enc.dev}; {len(todo)} texts to encode', flush=True)
    t = time.time()
    for i in range(0, len(todo), 2048):
        ks = todo[i:i + 2048]
        for k, v in zip(ks, enc([uniq[k] for k in ks], args.bs)):
            cache[k] = v.astype(np.float16)
        np.savez(cache_f, **cache)
        print(f'  encoded {i + len(ks)}/{len(todo)} [{time.time() - t:.0f}s]', flush=True)


def assemble(args):
    """Local: per-cut-off matrices aligned to user lists, read by rep.llm()."""
    with gzip.open(os.path.join(args.out, TEXTS), 'rt', encoding='utf-8') as f:
        per_cut = json.load(f)['cutoffs']
    q8 = os.path.join(args.out, f'{args.tag}_q8.npz')
    if os.path.exists(q8):
        # int8 copy made on the GPU machine to shrink the download (per-vector scale;
        # cosine to the fp16 original >= 0.9996 for every vector)
        z = np.load(q8)
        V = z['q'].astype(np.float32) * z['s']
        cache = {k: V[i] for i, k in enumerate(z['keys'].tolist())}
    else:
        cache = np.load(os.path.join(args.out, f'{args.tag}_cache.npz'))
    for c, m in per_cut.items():
        users = sorted(m)
        E = np.stack([cache[m[u]] for u in users]).astype(np.float32)
        np.save(os.path.join(args.out, f'{args.tag}_{c}.npy'), E)
        open(os.path.join(args.out, f'{args.tag}_{c}.users.txt'), 'w', encoding='utf-8').write('\n'.join(users))
        print(f'saved {args.tag}_{c}: {E.shape}', flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('stage', choices=['export', 'encode', 'assemble'])
    ap.add_argument('--data', default='data/FAR-Trans')
    ap.add_argument('--model', default='Qwen/Qwen3.5-0.8B')
    ap.add_argument('--tag', default='qwen35')
    ap.add_argument('--cutoffs', nargs='+', default=['2022-06-01', '2021-12-01', '2021-09-01', '2021-06-01', '2021-03-01'])
    ap.add_argument('--out', default='outputs/emb')
    ap.add_argument('--max-len', type=int, default=384)
    ap.add_argument('--bs', type=int, default=32)
    ap.add_argument('--threads', type=int, default=8)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    {'export': export, 'encode': encode, 'assemble': assemble}[args.stage](args)


if __name__ == '__main__':
    main()
''',
'src/customer_profile.py': r'''
"""Customer profile as known at the cut-off (leakage-safe rule or Shavrina's timestamp-only rule)."""
import pandas as pd


def profile_table(ci: pd.DataFrame, cutoff: pd.Timestamp, rule: str) -> pd.DataFrame:
    """One row per customer. rule 'approved': lastQuestionnaireDate < cutoff, not the 2000-01-01
    placeholder, row timestamp < cutoff; latest by (lastQuestionnaireDate, timestamp).
    rule 'timestamp': latest row with timestamp < cutoff (Shavrina's profile_at)."""
    d = str(cutoff.date())
    if rule == 'approved':
        ok = ci.lastQuestionnaireDate.astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')
        ci = ci[ok & (ci.lastQuestionnaireDate < d) & (ci.lastQuestionnaireDate != '2000-01-01')
                & (ci.timestamp.str[:10] < d)]
        ci = ci.sort_values(['customerID', 'lastQuestionnaireDate', 'timestamp'], kind='stable')
    elif rule == 'timestamp':
        ci = ci[ci.ts < cutoff].sort_values('ts', kind='stable')
    else:
        raise ValueError(rule)
    return ci.drop_duplicates('customerID', keep='last')
''',
'hw2/far-trans-llm/embedding_encoder.py': r'''
"""HW2 Encoder (DimSum, A. Shavrina): compressed LLM investor embeddings -> user-kNN scores.

Research question: how does compressing frozen-LLM investor representations to
128, 32, 8 or 2 dimensions (UMAP main, PCA side) change Top-10 nDCG@10 compared with
the uncompressed representation?

Pipeline (all inputs strictly before the cutoff):
  1. Each investor's as-of profile + pre-cutoff purchase history is serialised as text and
     encoded by frozen Qwen/Qwen3.5-0.8B (zero-shot PromptEOL prompt, last-token hidden
     state, 1024-d). Inference code: ../../src/embed.py (stages export/encode/assemble),
     run on a Colab T4 via ../../notebooks/encode_qwen35_colab.ipynb. This script loads the
     resulting matrix (--emb) and does NOT call the model again.
  2. The matrix (all users with a pre-cutoff buy, ~25k) is reduced with UMAP or PCA.
  3. For each evaluated user, the k=50 nearest investors in the reduced space vote for
     assets: score(i) = sum_j sim(u, j) * decayed_buys(j, i)  (half-life 365 days).
  4. The full eligible universe is ranked (repeat purchases allowed); ties and zero scores
     are ordered by the starter's popularity, then ISIN.

Same-information control: `behaviour_uknn` uses the identical neighbour rule on the
investors' decayed purchase vectors instead of LLM embeddings.
Hybrid ablation: `own_history_pop` (no neighbours) vs `hybrid_behaviour` vs `hybrid_llm_umap8`.

The provided starter (../far-trans) supplies users, cutoff, eligible assets, truth and
nDCG@10; its four baselines are written alongside via data_adapter.save_results.
"""
from __future__ import annotations

import json
import time
from collections import defaultdict
from datetime import date
from pathlib import Path

import numpy as np

from data_adapter import arguments, load, save_results

DIMS = (128, 32, 8, 2)


def decayed_matrix(events, users, item_ix, cutoff, half_life=365.0):
    """Users x items matrix of time-decayed pre-cutoff Buy counts (starter row policy)."""
    t0 = date.fromisoformat(cutoff)
    uix = {u: r for r, u in enumerate(users)}
    X = np.zeros((len(users), len(item_ix)), np.float32)
    for uid, evs in events.items():
        r = uix.get(uid)
        if r is None:
            continue
        for ts, isin, kind in evs:
            if kind == 'Buy' and ts < cutoff and isin in item_ix:
                age = (t0 - date.fromisoformat(ts)).days
                X[r, item_ix[isin]] += 0.5 ** (age / half_life)
    return X


def l2(M):
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)


def reduce(E, method, d, seed=42):
    """Return (Z, metric). Fit on all pre-cutoff users only (no evaluation labels)."""
    if method == 'none':
        return l2(E), 'cos'
    if method == 'pca':
        from sklearn.decomposition import PCA
        return l2(PCA(n_components=d, random_state=seed).fit_transform(l2(E))), 'cos'
    import umap
    Z = umap.UMAP(n_components=d, n_neighbors=30, min_dist=0.0, metric='cosine',
                  random_state=seed).fit_transform(E)
    return Z.astype(np.float32), 'euc'


def knn_scores(Z, metric, rows, X, k=50):
    """rows: query row indices. Returns len(rows) x items score matrix."""
    out = np.zeros((len(rows), X.shape[1]), np.float32)
    for n, r in enumerate(rows):
        if metric == 'cos':
            sims = Z @ Z[r]
        else:
            sims = 1.0 / (1.0 + np.linalg.norm(Z - Z[r], axis=1))
        sims[r] = -np.inf                       # never use the user as their own neighbour
        nn = np.argpartition(-sims, k)[:k]
        out[n] = sims[nn] @ X[nn]
    return out


def rank(scores, items, pop, eligible):
    """Full-universe top ten; ties broken by starter popularity, then ISIN."""
    s = {isin: float(scores[j]) for j, isin in enumerate(items) if isin in eligible}
    return sorted(s, key=lambda i: (-s[i], -pop.get(i, 0), i))[:10]


def main():
    parser = arguments(__doc__)
    parser.add_argument('--emb', type=Path, required=True, help='qwen35_<cutoff>.npy from src/embed.py')
    parser.add_argument('--k', type=int, default=50)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()
    t_all = time.perf_counter()
    bundle = load(args)
    starter, assets, history, truth, contexts, baseline, events, pop, manifest = bundle
    if any((args.output_dir / p).exists() for p in ('metrics.json', 'predictions.csv')):
        raise ValueError('Output directory already contains results; use a fresh directory')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    eval_users = manifest['users']
    items = sorted(assets)
    item_ix = {i: j for j, i in enumerate(items)}

    # Embedding matrix built from pre-cutoff information only (see src/embed.py).
    E = np.load(args.emb).astype(np.float32)
    emb_users = Path(str(args.emb).replace('.npy', '.users.txt')).read_text(encoding='utf-8').split('\n')
    assert len(emb_users) == len(E)
    missing = [u for u in eval_users if u not in set(emb_users)]
    if missing:
        raise ValueError(f'{len(missing)} evaluated users have no embedding')
    uix = {u: r for r, u in enumerate(emb_users)}
    rows = [uix[u] for u in eval_users]
    X = decayed_matrix(events, emb_users, item_ix, args.cutoff)
    timings = {'load_and_neighbour_matrix_seconds': time.perf_counter() - t_all}

    def score_block(name, Z, metric):
        t = time.perf_counter()
        S = knn_scores(Z, metric, rows, X, args.k)
        timings[name + '_knn_seconds'] = time.perf_counter() - t
        return S

    scores = {}
    scores['behaviour_uknn'] = score_block('behaviour_uknn', l2(X), 'cos')
    for method in ('none', 'umap', 'pca'):
        for d in ((0,) if method == 'none' else DIMS):
            name = 'llm_uknn_full' if method == 'none' else f'llm_uknn_{method}{d}'
            t = time.perf_counter()
            Z, metric = reduce(E, method, d, args.seed)
            timings[name + '_reduce_seconds'] = time.perf_counter() - t
            scores[name] = score_block(name, Z, metric)
            print(f'{name}: reduce {timings[name + "_reduce_seconds"]:.1f}s', flush=True)

    # hybrid: own decayed history + neighbour votes + popularity (fixed weights, set before this run)
    own = X[rows]
    pop_vec = np.array([pop.get(i, 0) for i in items], np.float32)
    nrm = lambda v: (v - v.min(1, keepdims=True)) / (v.max(1, keepdims=True) - v.min(1, keepdims=True) + 1e-9)
    P = np.tile(pop_vec / (pop_vec.max() + 1e-9), (len(rows), 1))
    scores['own_history_pop'] = nrm(own) + 0.1 * P
    scores['hybrid_behaviour'] = nrm(own) + 0.35 * nrm(scores['behaviour_uknn']) + 0.1 * P
    scores['hybrid_llm_umap8'] = nrm(own) + 0.35 * nrm(scores['llm_uknn_umap8']) + 0.1 * P

    predictions = defaultdict(dict)
    for name, S in scores.items():
        for n, uid in enumerate(eval_users):
            predictions[uid][name] = rank(S[n], items, pop, set(contexts[uid]['eligible_asset_ids']))
    timings['total_seconds'] = time.perf_counter() - t_all
    metadata = {
        'encoder_model': 'Qwen/Qwen3.5-0.8B', 'encoder_revision_local_snapshot': args.revision,
        'encoder_inference': 'frozen, zero-shot PromptEOL last-token, fp16 on Colab T4 '
                             '(notebooks/encode_qwen35_colab.ipynb); max_len 384; left padding',
        'prompt_template': 'This investor record:\\n"{text}"\\nmeans in one word:"',
        'embedding_file': str(args.emb), 'embedded_users': len(emb_users),
        'reducers': {'umap': 'n_neighbors=30, min_dist=0, metric=cosine, random_state=%d' % args.seed,
                     'pca': 'on L2-normalised vectors, random_state=%d' % args.seed},
        'knn': {'k': args.k, 'item_weight': 'Buy count decayed with 365-day half-life'},
        'hybrid_weights': 'own 1.0, neighbours 0.35, popularity 0.1 (fixed before this run)',
        'timings_seconds': timings, 'invalid_outputs': 0,
    }
    save_results(args, bundle, predictions, [], 'embedding_encoder', metadata)
    print(json.dumps({k: round(v, 1) for k, v in timings.items()}, indent=1))


if __name__ == '__main__':
    main()
''',
'hw2/far-trans-llm/reranker_encoder.py': r'''
"""HW2 Encoder inside a learned reranker (DimSum, A. Shavrina).

The compressed Qwen3.5 investor embedding (none / UMAP / PCA, d = 128, 32, 8, 2) produces a
user-kNN score for every asset; that score and its rank are added as features to a
LightGBM LambdaRank reranker (the team pipeline in ../../src). The control
`reranker_behaviour` is the identical reranker without the LLM feature, so the
difference between the two isolates what the LLM representation contributes.

Protocol (starter defaults): users, cutoff, eligible assets and nDCG@10 come from the
provided starter via data_adapter; the four provided baselines are written alongside.

Leakage rules:
  * Every feature for the test cutoff uses rows dated before 2021-06-01 only.
  * The reranker is trained on three earlier one-month windows (cutoffs 2021-03-01,
    2021-04-01, 2021-05-01; the last one is the early-stopping validation split). Their
    labels end on or before the test cutoff.
  * Training splits use the embedding file encoded at 2021-03-01 (texts built from rows
    before 2021-03-01), i.e. never later than their own cutoff. Users who first bought
    after 2021-03-01 have no embedding there and get a missing LLM feature (LightGBM
    handles NaN). The test split uses the file encoded at 2021-06-01.
  * Candidates = union of top-50 from own past buys, purchase-vector user-kNN, item-kNN,
    EASE, iALS and 90-day popularity; identical for every variant (LLM is a feature only).

python reranker_encoder.py --data-dir ../../data/FAR-Trans --output-dir ../outputs/reranker100 \
    --model Qwen/Qwen3.5-0.8B --revision 2fc06364715b967f1860aea9cf38778875588b17
"""
from __future__ import annotations

import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / 'src'))
from data import load as load_raw, make_split  # noqa: E402
from features import Signals, candidates, feature_frame  # noqa: E402

from data_adapter import arguments, load, save_results  # noqa: E402
from embedding_encoder import reduce  # noqa: E402

DIMS = (128, 32, 8, 2)
VARIANTS = [('none', 0)] + [('umap', d) for d in DIMS] + [('pca', d) for d in DIMS]
TRAIN = [('2021-05-01', '2021-06-01'), ('2021-04-01', '2021-05-01'), ('2021-03-01', '2021-04-01')]
DROP = ['user', 'item', 'label']


def vname(method, d):
    return 'reranker_llm_full' if method == 'none' else f'reranker_llm_{method}{d}'


def load_emb(emb_dir, cutoff):
    E = np.load(emb_dir / f'qwen35_{cutoff}.npy').astype(np.float32)
    users = (emb_dir / f'qwen35_{cutoff}.users.txt').read_text(encoding='utf-8').split()
    assert len(users) == len(E)
    return E, users


def reduced(E, tag, method, d, seed, cache):
    """Reduction is unsupervised and fit on all pre-cutoff users; cached across runs."""
    f = cache / f'{tag}_{method}{d}_s{seed}.npz'
    if f.exists():
        z = np.load(f)
        return z['Z'], str(z['metric'])
    Z, metric = reduce(E, method, d, seed)
    np.savez(f, Z=Z, metric=metric)
    return Z, metric


def llm_knn(sg, Z, metric, emb_users, query, k=50):
    """Neighbour score over items from the k most similar investors in Z (neighbours'
    365-day-decayed buys). NaN vector for users without an embedding at this cutoff."""
    pos = {u: i for i, u in enumerate(emb_users)}
    have = [u for u in sg.users if u in pos]
    Zs = Z[[pos[u] for u in have]]
    if metric == 'cos':
        Zs = Zs / (np.linalg.norm(Zs, axis=1, keepdims=True) + 1e-9)
    Xs = sg.Xd365[[sg.uix[u] for u in have]]
    hix = {u: i for i, u in enumerate(have)}
    out = {}
    nan = np.full(sg.Xd365.shape[1], np.nan, np.float32)
    for u in query:
        r = hix.get(u)
        if r is None:
            out[u] = nan
            continue
        sims = Zs @ Zs[r] if metric == 'cos' else 1.0 / (1.0 + np.linalg.norm(Zs - Zs[r], axis=1))
        sims[r] = -np.inf
        nn = np.argpartition(-sims, k)[:k]
        out[u] = sims[nn] @ Xs[nn]
    return out


def add_llm(df, scores, name):
    """Add score + rank columns for one LLM variant (rank NaN when the score is NaN)."""
    s_col, r_col = np.empty(len(df), np.float32), np.empty(len(df), np.float32)
    for u, idx in df.groupby('user', sort=False).indices.items():
        s = scores[u]
        items = df.item.values[idx]
        if np.isnan(s[0]):
            s_col[idx] = np.nan
            r_col[idx] = np.nan
        else:
            s_col[idx] = s[items]
            r_col[idx] = (-s).argsort().argsort()[items]
    df[name] = s_col
    df[name + '_rank'] = r_col


def main():
    parser = arguments(__doc__)
    parser.add_argument('--emb-dir', type=Path, default=ROOT / 'outputs' / 'emb')
    parser.add_argument('--k', type=int, default=50)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--lgb-seeds', type=int, default=3)
    parser.add_argument('--variants', default='all', help="'all' or 'behaviour' (control only)")
    args = parser.parse_args()
    t_all = time.perf_counter()
    bundle = load(args)
    starter, assets, history, truth, contexts, baseline, events, pop, manifest = bundle
    if any((args.output_dir / p).exists() for p in ('metrics.json', 'predictions.csv')):
        raise ValueError('Output directory already contains results; use a fresh directory')
    args.output_dir.mkdir(parents=True, exist_ok=True)
    cache = ROOT / 'hw2' / 'outputs' / 'cache'
    cache.mkdir(parents=True, exist_ok=True)
    eval_users = manifest['users']
    assert args.cutoff == '2021-06-01', 'training windows are set for the starter default cutoff'
    timings = {}
    variants = VARIANTS if args.variants == 'all' else []

    d = load_raw(str(args.data_dir))
    E_tr, U_tr = load_emb(args.emb_dir, '2021-03-01')
    E_te, U_te = load_emb(args.emb_dir, '2021-06-01')

    def frame(cutoff, end, users_fn, E, U, tag):
        t = time.perf_counter()
        sp = make_split(d, cutoff, end)
        sg = Signals(sp, customers=d['customers'])
        users = users_fn(sp, sg)
        bknn = sg.userknn(sg.Xd365, users, k=args.k)
        df = feature_frame(sg, users, candidates(sg, users, bknn, n=50), {'uknn': bknn})
        for method, dim in variants:
            Z, metric = reduced(E, tag, method, dim, args.seed, cache)
            add_llm(df, llm_knn(sg, Z, metric, U, users, args.k), 'llm')
            df = df.rename(columns={'llm': vname(method, dim), 'llm_rank': vname(method, dim) + '_rank'})
        df['label'] = [int(i in sp.truth.get(u, ())) for u, i in zip(df.user, df.item)]
        timings[f'features_{cutoff}_seconds'] = time.perf_counter() - t
        print(f'split {cutoff}: users {len(users)}, rows {len(df)}, positives {df.label.sum()} '
              f'[{time.perf_counter() - t:.0f}s]', flush=True)
        return sp, sg, df

    built = [frame(c, e, lambda sp, sg: sp.warm, E_tr, U_tr, 'qwen35_2021-03-01')[2] for c, e in TRAIN]
    end = str((pd.Timestamp(args.cutoff) + pd.Timedelta(days=args.horizon_days)).date())
    sp, sg, test = frame(args.cutoff, end, lambda sp, sg: [u for u in eval_users if u in sg.uix],
                         E_te, U_te, 'qwen35_2021-06-01')
    no_signal = [u for u in eval_users if u not in sg.uix]
    print(f'{len(no_signal)} evaluated users without an eligible pre-cutoff buy -> popularity order')

    import lightgbm as lgb
    params = dict(objective='lambdarank', metric='ndcg', eval_at=[10], learning_rate=0.03,
                  num_leaves=31, min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8,
                  bagging_freq=1, lambdarank_truncation_level=20, verbose=-1, seed=0)

    def ds(df):
        df = df.sort_values('user', kind='stable')
        return df, df.groupby('user', sort=False).size().values

    va, gva = ds(built[0])
    trn, gtr = ds(pd.concat(built[1:]))
    full, gfull = ds(pd.concat(built))
    base = [c for c in test.columns if c not in DROP and not c.startswith('reranker_llm_')]
    llm_cols = {vname(m, dd): [vname(m, dd), vname(m, dd) + '_rank'] for m, dd in variants}
    configs = [('reranker_behaviour', base)] + [(n, base + cols) for n, cols in llm_cols.items()]

    pop_order = sorted(pop, key=lambda i: (-pop[i], i))
    predictions = defaultdict(dict)
    info = {}
    for name, feats in configs:
        t = time.perf_counter()
        m = lgb.train(params, lgb.Dataset(trn[feats], trn.label, group=gtr), 2000,
                      valid_sets=[lgb.Dataset(va[feats], va.label, group=gva)],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        best = max(m.best_iteration, 1)
        score = np.zeros(len(test))
        for s in range(args.lgb_seeds):  # seed-averaged refit on all three training windows
            ms = lgb.train({**params, 'seed': s}, lgb.Dataset(full[feats], full.label, group=gfull), best)
            score += ms.predict(test[feats]) / args.lgb_seeds
        imp = pd.Series(ms.feature_importance('gain'), feats)
        info[name] = {'best_iteration': best, 'valid_ndcg10_candidates': m.best_score['valid_0']['ndcg@10'],
                      'llm_feature_gain_share': float(imp[[f for f in feats if f.startswith('reranker_llm_')]].sum()
                                                      / imp.sum())}
        tt = test.assign(score=score)
        for u, g in tt.groupby('user', sort=False):
            elig = set(contexts[u]['eligible_asset_ids'])
            order = [sp.items[i] for i in g.item.values[np.argsort(-g.score.values, kind='stable')]]
            top = [i for i in order if i in elig][:10]
            for i in pop_order:          # pad with starter popularity if fewer than ten candidates
                if len(top) == 10:
                    break
                if i in elig and i not in top:
                    top.append(i)
            predictions[u][name] = top
        for u in no_signal:
            elig = set(contexts[u]['eligible_asset_ids'])
            predictions[u][name] = [i for i in pop_order if i in elig][:10]
        timings[name + '_train_predict_seconds'] = time.perf_counter() - t
        print(f'{name}: best it {best}, valid {info[name]["valid_ndcg10_candidates"]:.4f}, '
              f'LLM gain share {info[name]["llm_feature_gain_share"]:.3f} [{time.perf_counter() - t:.0f}s]',
              flush=True)

    timings['total_seconds'] = time.perf_counter() - t_all
    metadata = {
        'encoder_model': 'Qwen/Qwen3.5-0.8B', 'encoder_revision_local_snapshot': args.revision,
        'encoder_inference': 'frozen, zero-shot PromptEOL last-token, fp16 on Colab T4 '
                             '(notebooks/encode_qwen35_colab.ipynb); max_len 384; left padding',
        'prompt_template': 'This investor record:\\n"{text}"\\nmeans in one word:"',
        'embedding_files': {'train': 'qwen35_2021-03-01.npy', 'test': 'qwen35_2021-06-01.npy'},
        'reducers': {'umap': f'n_neighbors=30, min_dist=0, metric=cosine, random_state={args.seed}',
                     'pca': f'on L2-normalised vectors, random_state={args.seed}'},
        'reranker': {'model': 'LightGBM lambdarank', 'params': params, 'lgb_seeds': args.lgb_seeds,
                     'train_windows': TRAIN, 'validation_window': TRAIN[0]},
        'candidate_policy': 'union of top-50 per source: own past buys, purchase-vector user-kNN, item-kNN, '
                            'EASE, iALS, 90-day popularity; filtered to the starter eligible set; padded '
                            'with starter popularity',
        'users_without_signal': len(no_signal), 'variants': info,
        'timings_seconds': timings, 'invalid_outputs': 0,
    }
    save_results(args, bundle, predictions, [], 'reranker_encoder', metadata)
    print({k: round(v, 1) for k, v in timings.items()})


if __name__ == '__main__':
    main()
''',
}
for name, text in MODULES.items():
    (WORK / name).parent.mkdir(parents=True, exist_ok=True)
    (WORK / name).write_text(text, encoding='utf-8')
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(HW2))
import data as D, embed as EM, customer_profile as PR
print('modules written: src/data.py, src/features.py, src/embed.py, src/customer_profile.py, and her two HW2 scripts')

modules written: src/data.py, src/features.py, src/embed.py, src/customer_profile.py, and her two HW2 scripts


In [7]:
# Her reranker script, with exactly three edits (each must match once):
#   (1) it uses the approved profile rule through PROFILE_TABLE instead of the raw customer table;
#   (2) it adds one control variant, the same profile fields as one-hot vectors (no LLM);
#   (3) that control's feature is named reranker_profile_onehot_knn and is excluded from the base features.
RR = HW2 / 'reranker_encoder.py'
rr_src = RR.read_text(encoding='utf-8')
EDITS = [
    ("VARIANTS = [('none', 0)] + [('umap', d) for d in DIMS] + [('pca', d) for d in DIMS]",
     "VARIANTS = [('none', 0)] + [('umap', d) for d in DIMS] + [('pca', d) for d in DIMS] + [('onehot', 0)]"),
    ("def vname(method, d):\n    return 'reranker_llm_full' if method == 'none' else f'reranker_llm_{method}{d}'",
     "def vname(method, d):\n    if method == 'onehot':\n        return 'reranker_profile_onehot_knn'\n    return 'reranker_llm_full' if method == 'none' else f'reranker_llm_{method}{d}'"),
    ("            Z, metric = reduced(E, tag, method, dim, args.seed, cache)\n            add_llm(df, llm_knn(sg, Z, metric, U, users, args.k), 'llm')",
     "            Ex = ONEHOT[tag] if method == 'onehot' else E\n            Z, metric = reduced(Ex, 'onehot_' + tag if method == 'onehot' else tag, 'none' if method == 'onehot' else method, dim, args.seed, cache)\n            add_llm(df, llm_knn(sg, Z, metric, U, users, args.k), 'llm')"),
    ("    base = [c for c in test.columns if c not in DROP and not c.startswith('reranker_llm_')]",
     "    base = [c for c in test.columns if c not in DROP and not c.startswith(('reranker_llm_', 'reranker_profile_'))]"),
    ("        sg = Signals(sp, customers=d['customers'])",
     "        sg = Signals(sp, customers=PROFILE_TABLE(d['customers'], pd.Timestamp(cutoff), PROFILE_RULE))"),
]
for old, new in EDITS:
    assert rr_src.count(old) == 1, ('edit does not match once', old[:70], rr_src.count(old))
    rr_src = rr_src.replace(old, new)
RR.write_text(rr_src, encoding='utf-8')
print('reranker_encoder.py edited:', len(EDITS), 'edits applied')

reranker_encoder.py edited: 5 edits applied


In [8]:
d = D.load(str(DATA_DIR))
TEXTS = {}
for c in TAG_DATES:
    end_c = str((pd.Timestamp(c) + pd.DateOffset(months=1)).date())
    sp_c = D.make_split(d, c, end_c)
    ci_c = PR.profile_table(d['customers'], pd.Timestamp(c), PROFILE_RULE)
    TEXTS[c] = EM.user_texts(sp_c, ci_c)
    print(f'{c}: investors with text {len(TEXTS[c])}, items {len(sp_c.items)}, profile rule {PROFILE_RULE}')

2021-03-01: investors with text 23398, items 757, profile rule approved
2021-06-01: investors with text 25126, items 767, profile rule approved


In [9]:
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV, '| torch', torch.__version__)
CACHE = WORK / 'text_embeddings_cache.npz'
cache = dict(np.load(CACHE)) if CACHE.exists() else {}
uniq = {}
for c in TAG_DATES:
    for t in TEXTS[c].values():
        uniq.setdefault(hashlib.md5(t.encode()).hexdigest(), t)
todo = sorted((h for h in uniq if h not in cache), key=lambda h: len(uniq[h]))
print('unique texts:', len(uniq), '| to encode now:', len(todo))
if todo:
    enc = EM.Encoder(MODEL, MAX_LEN, threads=4)
    t_enc = time.time()
    for i in range(0, len(todo), 2048):
        ks = todo[i:i + 2048]
        for k, v in zip(ks, enc([uniq[k] for k in ks], BATCH)):
            cache[k] = v.astype(np.float16)
        np.savez(CACHE, **cache)
        print(f'encoded {min(i + 2048, len(todo))}/{len(todo)} [{time.time() - t_enc:.0f}s]', flush=True)
    encode_seconds = time.time() - t_enc
    del enc
    if DEV == 'cuda':
        torch.cuda.empty_cache()
else:
    encode_seconds = 0.0
for c in TAG_DATES:
    users = sorted(TEXTS[c])
    E_c = np.stack([cache[hashlib.md5(TEXTS[c][u].encode()).hexdigest()] for u in users]).astype(np.float32)
    np.save(EMB / f'qwen35_{c}.npy', E_c)
    (EMB / f'qwen35_{c}.users.txt').write_text('\n'.join(users), encoding='utf-8')
    print(f'saved qwen35_{c}: {E_c.shape}')

device: cuda | torch 2.11.0+cu130
unique texts: 26211 | to encode now: 26211


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


encoded 2048/26211 [166s]
encoded 4096/26211 [232s]
encoded 6144/26211 [320s]
encoded 8192/26211 [410s]
encoded 10240/26211 [502s]
encoded 12288/26211 [601s]
encoded 14336/26211 [714s]
encoded 16384/26211 [836s]
encoded 18432/26211 [980s]
encoded 20480/26211 [1129s]
encoded 22528/26211 [1317s]
encoded 24576/26211 [1531s]
encoded 26211/26211 [1711s]
saved qwen35_2021-03-01: (23398, 1024)
saved qwen35_2021-06-01: (25126, 1024)


In [10]:
# One-hot profile control: customer type, risk level, capacity (Predicted_ prefix removed); no LLM.
ONEHOT = {}
for c in TAG_DATES:
    users = sorted(TEXTS[c])
    prof = PR.profile_table(d['customers'], pd.Timestamp(c), PROFILE_RULE).set_index('customerID')
    cols = {'type': [], 'risk': [], 'capacity': []}
    for u in users:
        if u in prof.index:
            r_ = prof.loc[u]
            cols['type'].append(str(r_['customerType']))
            cols['risk'].append(str(r_['riskLevel']).replace('Predicted_', ''))
            cols['capacity'].append(str(r_['investmentCapacity']).replace('Predicted_', ''))
        else:
            for k in cols:
                cols[k].append('no_profile')
    M = pd.get_dummies(pd.DataFrame(cols, index=users)).astype(np.float32).values
    ONEHOT[f'qwen35_{c}'] = M
    print(f'one-hot profile matrix {c}: {M.shape}; users without a profile row: {sum(1 for u in users if u not in prof.index)}')

one-hot profile matrix 2021-03-01: (23398, 18); users without a profile row: 11764
one-hot profile matrix 2021-06-01: (25126, 18); users without a profile row: 11356


In [12]:
import embedding_encoder as EE
mods_argv = ['embedding_encoder.py', '--data-dir', str(DATA_DIR), '--output-dir', str(OUT_A),
             '--model', MODEL, '--revision', MODEL_REVISION, '--cutoff', CUTOFF,
             '--emb', str(EMB / 'qwen35_2021-06-01.npy'), '--max-users', str(MAX_USERS)]
sys.argv = mods_argv
t_a = time.time()
EE.main()
print(f'stage A finished in {time.time() - t_a:.0f}s -> {OUT_A}')

llm_uknn_full: reduce 0.1s


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


llm_uknn_umap128: reduce 201.3s


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


llm_uknn_umap32: reduce 124.5s


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


llm_uknn_umap8: reduce 105.8s


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


llm_uknn_umap2: reduce 113.8s
llm_uknn_pca128: reduce 2.7s
llm_uknn_pca32: reduce 2.0s
llm_uknn_pca8: reduce 1.3s
llm_uknn_pca2: reduce 1.0s
behaviour_uknn: nDCG@10=0.0400
cbf: nDCG@10=0.0201
hybrid_behaviour: nDCG@10=0.0371
hybrid_llm_umap8: nDCG@10=0.0379
latest_buy: nDCG@10=0.0277
latest_buy_or_sell: nDCG@10=0.0271
llm_uknn_full: nDCG@10=0.0323
llm_uknn_pca128: nDCG@10=0.0331
llm_uknn_pca2: nDCG@10=0.0262
llm_uknn_pca32: nDCG@10=0.0314
llm_uknn_pca8: nDCG@10=0.0241
llm_uknn_umap128: nDCG@10=0.0295
llm_uknn_umap2: nDCG@10=0.0325
llm_uknn_umap32: nDCG@10=0.0334
llm_uknn_umap8: nDCG@10=0.0298
own_history_pop: nDCG@10=0.0384
popularity: nDCG@10=0.0286
{
 "load_and_neighbour_matrix_seconds": 10.3,
 "behaviour_uknn_knn_seconds": 0.3,
 "llm_uknn_full_reduce_seconds": 0.1,
 "llm_uknn_full_knn_seconds": 0.4,
 "llm_uknn_umap128_reduce_seconds": 201.3,
 "llm_uknn_umap128_knn_seconds": 0.6,
 "llm_uknn_umap32_reduce_seconds": 124.5,
 "llm_uknn_umap32_knn_seconds": 0.2,
 "llm_uknn_umap8_reduce_se

In [13]:
import reranker_encoder as RE
RE.PROFILE_TABLE = PR.profile_table
RE.PROFILE_RULE = PROFILE_RULE
RE.ONEHOT = ONEHOT
sys.argv = ['reranker_encoder.py', '--data-dir', str(DATA_DIR), '--output-dir', str(OUT_B),
            '--model', MODEL, '--revision', MODEL_REVISION, '--cutoff', CUTOFF,
            '--emb-dir', str(EMB), '--max-users', str(MAX_USERS)]
t_b = time.time()
RE.main()
print(f'stage B finished in {time.time() - t_b:.0f}s -> {OUT_B}')

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compa

split 2021-05-01: users 1714, rows 169148, positives 3399 [634s]
split 2021-04-01: users 1772, rows 175470, positives 3505 [67s]
split 2021-03-01: users 1841, rows 180759, positives 3670 [67s]


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compa

split 2021-06-01: users 100, rows 11289, positives 18 [571s]
0 evaluated users without an eligible pre-cutoff buy -> popularity order
reranker_behaviour: best it 167, valid 0.6103, LLM gain share 0.000 [52s]
reranker_llm_full: best it 142, valid 0.6084, LLM gain share 0.004 [47s]
reranker_llm_umap128: best it 237, valid 0.6098, LLM gain share 0.006 [71s]
reranker_llm_umap32: best it 326, valid 0.6105, LLM gain share 0.010 [94s]
reranker_llm_umap8: best it 225, valid 0.6113, LLM gain share 0.007 [67s]
reranker_llm_umap2: best it 246, valid 0.6104, LLM gain share 0.006 [73s]
reranker_llm_pca128: best it 145, valid 0.6082, LLM gain share 0.004 [47s]
reranker_llm_pca32: best it 256, valid 0.6090, LLM gain share 0.006 [75s]
reranker_llm_pca8: best it 182, valid 0.6093, LLM gain share 0.004 [56s]
reranker_llm_pca2: best it 212, valid 0.6083, LLM gain share 0.006 [65s]
reranker_profile_onehot_knn: best it 159, valid 0.6087, LLM gain share 0.000 [52s]
cbf: nDCG@10=0.0201
latest_buy: nDCG@10=0.

In [14]:
def load_run(out):
    metrics = json.loads((out / 'metrics.json').read_text(encoding='utf-8'))
    manifest = json.loads((out / 'run_manifest.json').read_text(encoding='utf-8'))
    return metrics, manifest

mA, mfA = load_run(OUT_A)
mB, mfB = load_run(OUT_B)
timings = {**mfA['runtime'].get('timings_seconds', {}), **mfB['runtime'].get('timings_seconds', {})}

def runtime_for(name):
    keys = [k for k in timings if k.startswith(name + '_') and k.endswith(('_seconds'))]
    if name in ('popularity', 'cbf', 'latest_buy', 'latest_buy_or_sell'):
        return 'shared (whole starter run)'
    parts = {k: round(timings[k], 1) for k in keys if k.endswith(('_train_predict_seconds', '_reduce_seconds', '_knn_seconds'))}
    return parts if parts else 'n/a'

rows = []
for name, m in {**mA, **mB}.items():
    rows.append({'method': name, 'users': m['users'], 'users_with_future_purchases': m['nonempty_truth_users'],
                 'ndcg_all': m['ndcg_at_10'], 'ndcg_nonempty': m['ndcg_at_10_nonempty'],
                 'runtime_seconds': runtime_for(name), 'failed_cases': 0})
TAB_HW2 = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 120)
print('HW2 table (starter protocol, cutoff 2021-06-01, 30 days, users =', MAX_USERS, ')')
print(TAB_HW2.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('\nShared feature building per stage-B split (seconds):', {k: round(v, 1) for k, v in timings.items() if k.startswith('features_')})
print('Stage A total seconds:', round(timings.get('total_seconds', 0), 1))

HW2 table (starter protocol, cutoff 2021-06-01, 30 days, users = 100 )
                     method  users  users_with_future_purchases  ndcg_all  ndcg_nonempty                                                                 runtime_seconds  failed_cases
             behaviour_uknn    100                           11    0.0400         0.3638                                             {'behaviour_uknn_knn_seconds': 0.3}             0
                        cbf    100                           11    0.0201         0.1827                                                      shared (whole starter run)             0
           hybrid_behaviour    100                           11    0.0371         0.3374                                                                             n/a             0
           hybrid_llm_umap8    100                           11    0.0379         0.3447                                                                             n/a             0
              

In [16]:
per_b = pd.DataFrame(json.loads((OUT_B / 'per_user.json').read_text(encoding='utf-8')))
pred_b = pd.read_csv(OUT_B / 'predictions.csv')
tx = pd.read_csv(DATA_DIR / 'transactions.csv', dtype=str, keep_default_na=False).drop_duplicates()
tx = tx[tx.transactionID.astype(int) >= 0]
sys.path.insert(0, str(STARTER))
import starter as S
assets_ev = S.load_data(DATA_DIR, CUTOFF, HORIZON_DAYS)[0]
fut = tx[(tx.transactionType == 'Buy') & (tx.timestamp >= CUTOFF) & (tx.timestamp < END) & (tx.ISIN.isin(set(assets_ev)))]
truth_ind = fut.groupby('customerID').ISIN.apply(set).to_dict()

def ndcg_indep(ranking, truth_set):
    if not truth_set:
        return 0.0
    dcg = sum(1 / np.log2(r + 1) for r, i in enumerate(ranking[:10], 1) if i in truth_set)
    idcg = sum(1 / np.log2(r + 1) for r in range(1, min(10, len(truth_set)) + 1))
    return dcg / idcg

nonempty = sorted(u for u in pred_b.user_id.unique() if truth_ind.get(u))
print(f'starter users with a future purchase: {len(nonempty)}')
check_methods = ['reranker_llm_full', 'reranker_behaviour', 'reranker_profile_onehot_knn', 'latest_buy']
out_rows = []
for m in check_methods:
    sub = pred_b[pred_b.method == m]
    for u in nonempty:
        ranking = sub[sub.user_id == u].sort_values('rank').ISIN.tolist()
        hits = [r for r, i in enumerate(ranking, 1) if i in truth_ind[u]]
        saved = per_b[(per_b.user_id == u) & (per_b.method == m)].ndcg_at_10
        saved = float(saved.iloc[0]) if len(saved) else float('nan')
        mine = ndcg_indep(ranking, truth_ind[u])
        out_rows.append({'method': m, 'user': u, 'truth_size': len(truth_ind[u]), 'hit_ranks': hits,
                         'saved_ndcg': round(saved, 6), 'independent_ndcg': round(mine, 6),
                         'match': abs(saved - mine) < 1e-6})
VER = pd.DataFrame(out_rows)
print(VER.to_string(index=False))
print('\nall match:', bool(VER.match.all()), '| rows checked:', len(VER))

starter users with a future purchase: 11
                     method                 user  truth_size     hit_ranks  saved_ndcg  independent_ndcg  match
          reranker_llm_full 0E72D1BF99E858EEE1AA           1            []    0.000000          0.000000   True
          reranker_llm_full 29C0592F254DB2430C00           1            []    0.000000          0.000000   True
          reranker_llm_full 5087D23F907F90F3DDD1           1           [5]    0.386853          0.386853   True
          reranker_llm_full 5180B50A10AAD1C7D0F8           2           [4]    0.264068          0.264068   True
          reranker_llm_full 6B15913A38A56CB253E0           1           [7]    0.333333          0.333333   True
          reranker_llm_full 6BAA2D5FBCCEAB57C473           1           [5]    0.386853          0.386853   True
          reranker_llm_full 6FA5BF73C309CE8E9ED5           6     [3, 4, 6]    0.389414          0.389414   True
          reranker_llm_full 870E814DB274CFA95600           2   

In [17]:
if MAX_USERS >= 100000:
    pre = tx[tx.timestamp < CUTOFF]
    n_tx = pre.groupby('customerID').size().to_dict()
    n_isin = pre.groupby('customerID').ISIN.nunique().to_dict()
    S_assets, S_profiles, S_history, S_truth, _, _ = S.load_data(DATA_DIR, CUTOFF, HORIZON_DAYS)
    has_hist = {u for u, c in S_history.items() if c}
    evaluated = set(tx[(tx.transactionType == 'Buy') & (tx.timestamp >= CUTOFF) & (tx.timestamp < END)].customerID)
    warm = sorted(u for u in evaluated if u in has_hist)
    cold = sorted(evaluated - set(warm))
    ci_raw = pd.read_csv(DATA_DIR / 'customer_information.csv', dtype=str, keep_default_na=False)
    okd = ci_raw.lastQuestionnaireDate.str.match(r'^\d{4}-\d{2}-\d{2}$')
    adm_set = set(ci_raw[okd & (ci_raw.lastQuestionnaireDate < CUTOFF) & (ci_raw.lastQuestionnaireDate != '2000-01-01')
                         & (ci_raw.timestamp.str[:10] < CUTOFF)].customerID)
    cold_profile = [u for u in cold if u in adm_set]
    cold_noinfo = [u for u in cold if u not in adm_set]
    print(f'warm users (pre-cutoff eligible Buy, evaluated): {len(warm)} | cold: {len(cold)} | '
          f'cold-profile (approved, dropped as subgroup): {len(cold_profile)} | cold-no-information: {len(cold_noinfo)}')

    uni = list(S_assets)
    popc = S.popularity_scores(S_history, uni)
    POP10 = S.order({i: popc[i] for i in uni})[:10]
    per_A = pd.DataFrame(json.loads((OUT_A / 'per_user.json').read_text(encoding='utf-8')))
    per_B = pd.DataFrame(json.loads((OUT_B / 'per_user.json').read_text(encoding='utf-8')))
    piv_A = per_A.pivot_table(index='user_id', columns='method', values='ndcg_at_10')
    piv_B = per_B.pivot_table(index='user_id', columns='method', values='ndcg_at_10')
    piv = pd.concat([piv_A, piv_B.drop(columns=[c for c in piv_B.columns if c in piv_A.columns])], axis=1)
    cold_value = {u: S.ndcg10(POP10, S_truth.get(u, set())) for u in cold_noinfo}
    for u in cold_noinfo:
        piv.loc[u, :] = cold_value[u]
    missing = [u for u in warm if u not in piv.index]
    assert not missing, f'{len(missing)} warm users missing from the outputs'

    COH = {
        'warm_all': warm,
        'sparse_light_1_5_tx': [u for u in warm if n_tx.get(u, 0) <= 5],
        'medium_6_58_tx': [u for u in warm if 6 <= n_tx.get(u, 0) <= 58],
        'dense_heavy_gt58_tx': [u for u in warm if n_tx.get(u, 0) > 58],
        'div_le3_isin': [u for u in warm if n_isin.get(u, 0) <= 3],
        'div_4_6_isin': [u for u in warm if 4 <= n_isin.get(u, 0) <= 6],
        'div_7_12_isin': [u for u in warm if 7 <= n_isin.get(u, 0) <= 12],
        'div_gt12_isin': [u for u in warm if n_isin.get(u, 0) > 12],
        'cold_noinfo_control': cold_noinfo,
    }
    COH_TAB = pd.DataFrame({c: piv.loc[users].mean() for c, users in COH.items()})
    COH_TAB.insert(0, 'users', pd.Series({c: len(u) for c, u in COH.items()}))
    print('nDCG@10 by cohort (all users in cohort, empty truth = 0); cold-no-information = starter popularity list')
    print(COH_TAB.round(4).to_string())

    def boot_ci(dd, B=10000):
        idx = np.random.default_rng(0).integers(0, len(dd), (B, len(dd)))
        bm = dd[idx].mean(1)
        return float(dd.mean()), float(np.quantile(bm, 0.025)), float(np.quantile(bm, 0.975)), float(np.quantile(bm, 0.05 / 4))

    CTRL = {'stage_A': 'llm_uknn_full', 'stage_B': 'reranker_llm_full'}
    FAMILIES = {'stage_A': ['llm_uknn_umap%d' % x for x in DIMS] + ['llm_uknn_pca%d' % x for x in DIMS] + ['behaviour_uknn'],
                'stage_B': ['reranker_llm_umap%d' % x for x in DIMS] + ['reranker_llm_pca%d' % x for x in DIMS]
                            + ['reranker_behaviour', 'reranker_profile_onehot_knn']}
    rows = []
    for stage, methods in FAMILIES.items():
        ctrl = CTRL[stage]
        for cname, users in COH.items():
            for m in methods:
                if m not in piv.columns:
                    continue
                dd = (piv.loc[users, m] - piv.loc[users, ctrl]).values
                mean, lo, hi, lb = boot_ci(dd)
                rows.append({'stage': stage, 'cohort': cname, 'method': m, 'users': len(users),
                             'diff_vs_full': mean, 'ci95_low': lo, 'ci95_high': hi, 'bonferroni_lower': lb})
    PAIR = pd.DataFrame(rows)
    print('\nPaired differences vs uncompressed control, all cohorts (descriptive CIs)')
    print(PAIR.round(4).to_string(index=False))

    print('\nNon-inferiority on warm users, margin -0.01, Bonferroni over the four dimensions of each family:')
    ni_rows = []
    for stage in ('stage_A', 'stage_B'):
        for fam in ('umap', 'pca'):
            sub = PAIR[(PAIR.stage == stage) & (PAIR.cohort == 'warm_all') & PAIR.method.str.contains(fam)]
            for _, r_ in sub.iterrows():
                ni_rows.append({'stage': stage, 'method': r_.method, 'diff': r_.diff_vs_full,
                                'bonferroni_lower': r_.bonferroni_lower, 'non_inferior': r_.bonferroni_lower > -0.01})
    print(pd.DataFrame(ni_rows).round(4).to_string(index=False))
    print('\ncold-no-information control: max |difference| to the uncompressed control =',
          round(max(abs(piv.loc[u, m] - piv.loc[u, CTRL['stage_B']]) for u in cold_noinfo for m in FAMILIES['stage_B'] if m in piv.columns), 6))
    COH_TAB.to_csv(OUT_B / 'cohort_table.csv')
    PAIR.to_csv(OUT_B / 'cohort_paired.csv', index=False)
else:
    print('cohort analysis runs when MAX_USERS = 100000 (full warm cohort)')

cohort analysis runs when MAX_USERS = 100000 (full warm cohort)


In [19]:
protocol = {
    'cutoff': CUTOFF, 'end': END, 'max_users': MAX_USERS, 'profile_rule': PROFILE_RULE,
    'deviations_from_shavrina_hw2': ['profile rule (approved leakage fix)',
                                     'added profile one-hot control variant (reranker_profile_onehot_knn)'],
    'faithful_to_shavrina_hw2': ['starter data loader via data_adapter', 'universe from make_split (her code)',
                                 'raw transaction rows (her code)', 'three training windows 2021-03..05',
                                 'stale 2021-03-01 embedding for training windows', 'purchase-kNN candidates',
                                 'default UMAP layout (embedding_encoder.reduce)', 'LightGBM params and seeds'],
    'starter_repo_commit': REPO_COMMIT, 'model': MODEL, 'model_revision': MODEL_REVISION,
    'max_len': MAX_LEN, 'batch': BATCH, 'device': DEV, 'python': platform.python_version(),
    'encode_seconds_this_session': encode_seconds,
    'versions': {p: _ver(p) for p in ('torch', 'transformers', 'lightgbm', 'umap-learn', 'scikit-learn', 'pandas', 'numpy')},
}
(OUT_B / 'notebook_protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
print('saved', OUT_B / 'notebook_protocol.json')

saved /kaggle/working/outputs/stageB_reranker_approved_100/notebook_protocol.json
